***Welcome! Hello!***

We are glad you're here 👏! And are excited to get you rolling 🛣!

This notebook exists to enable you scan J1939.

It builds on the things done in `easy_*.ipynb` so if you haven't used those yet, please start there.

This is not an 'easy' notebook so we may ask you to edit the code in the cells.

First we'll do some imports and define some functions we will use later

In [ ]:
import getopt
import sys
import signal
import re
import threading
import time

from IPython.core.interactiveshell import InteractiveShell
from scapy.contrib.automotive.scanner.enumerator import ServiceEnumerator

InteractiveShell.ast_node_interactivity = 'all'

import ipywidgets as widgets
from ipywidgets import interact, interact_manual, Textarea

import binascii
import pandas as pd
import qgrid
from collections import defaultdict

import platform
IS_WINDOWS = platform.system() == 'Windows'
DEFAULT_INTERFACE = 'candle' if IS_WINDOWS else 'socketcan'
DEFAULT_CHANNEL = '0' if IS_WINDOWS else 'can0'

qgrid.enable()

In [ ]:
from scapy.all import *

load_layer("can")
conf.contribs['CANSocket'] = {'use-python-can': True}
load_contrib('cansocket')
load_contrib('isotp')
load_contrib('automotive.uds')
load_contrib('automotive.uds_scan')

In [ ]:
from tabulate import tabulate

def format_sa_range(actual_sas):
    if not actual_sas:
        return ""
    actual_sas = sorted(list(set(actual_sas)))
    ranges = []
    start = actual_sas[0]
    prev = actual_sas[0]
    for n in actual_sas[1:]:
        if n == prev + 1:
            prev = n
        else:
            if start == prev:
                ranges.append(f"0x{start:02x}")
            else:
                ranges.append(f"0x{start:02x}..0x{prev:02x}")
            start = n
            prev = n
    if start == prev:
        ranges.append(f"0x{start:02x}")
    else:
        ranges.append(f"0x{start:02x}..0x{prev:02x}")
    
    return "[" + ", ".join(ranges) + "]"

def get_all_sas_for_method(info, target_method):
    """
    Extracts all successful scanner SAs for a specific method from the result info.
    Inspects both the recorded 'src_addrs' and the 'packets' list (for PDU1 response DAs).
    """
    sas = set()
    methods = info.get('methods', [])
    src_addrs = info.get('src_addrs', [])
    packets = info.get('packets', [])
    
    for i, m in enumerate(methods):
        if m == target_method:
            # 1. Check the recorded src_addr (successes recorded by scanner)
            if i < len(src_addrs):
                val = src_addrs[i]
                if val is not None:
                    if isinstance(val, (list, tuple)):
                        for v in val:
                            if v is not None:
                                sas.add(v)
                    else:
                        sas.add(val)
            
            # 2. Inspect all packets for this method for PDU1 responses
            if i < len(packets):
                for p in packets[i]:
                    try:
                        # Extraction: In J1939 PDU1 format responses (PF < 240),
                        # the Destination Address (DA) in the PS field is the scanner's SA.
                        if hasattr(p, 'pf') and p.pf < 240:
                            sas.add(p.ps)
                        elif hasattr(p, 'identifier'):
                            # bits 8-15 are PS (DA when PDU1)
                            # bits 16-23 are PF
                            pf = (p.identifier >> 16) & 0xFF
                            if pf < 240:
                                ps = (p.identifier >> 8) & 0xFF
                                sas.add(ps)
                    except:
                        pass
    # 0xff is broadcast DA, shouldn't be a scanner SA in a PDU1 response
    if 0xff in sas:
        sas.remove(0xff)
    return sorted(list(sas))

def get_method_summary(info, method_name):
    """
    Summarizes all successful SAs for a specific method into a range-based string.
    """
    sas = get_all_sas_for_method(info, method_name)
    if not sas:
        return "using broadcast"
    return "from SAs " + format_sa_range(sas)

def merge_scan_results(*result_dicts, passive_results=None):
    """
    Combines multiple J1939 scan result dictionaries into a single master summary.
    Correctly handles 'methods', 'src_addrs', and 'packets' lists.
    Optional 'passive_results' is a list of SAs found via passive scanning.
    """
    merged = {}
    
    # 1. Merge all active result dictionaries
    for res_dict in result_dicts:
        if not res_dict:
            continue
        for da, info in res_dict.items():
            if da not in merged:
                # Initialize with copies to avoid mutating inputs
                merged[da] = {
                    'methods': list(info.get('methods', [])),
                    'src_addrs': list(info.get('src_addrs', [])),
                    'packets': list(info.get('packets', []))
                }
            else:
                merged[da]['methods'].extend(info.get('methods', []))
                merged[da]['src_addrs'].extend(info.get('src_addrs', []))
                merged[da]['packets'].extend(info.get('packets', []))
                
    # 2. Merge passive results (list of SAs)
    if passive_results:
        for da in passive_results:
            if da not in merged:
                merged[da] = {'methods': ['passive'], 'src_addrs': [None], 'packets': [[]]}
            elif 'passive' not in merged[da]['methods']:
                merged[da]['methods'].append('passive')
                merged[da]['src_addrs'].append(None)
                merged[da]['packets'].append([])
                
    return merged

def generate_markdown_table(sa_dict):
    """
    Generates a Markdown table using the tabulate library.
    Ingests: {SA_INT: {'methods': [...], 'src_addrs': [...], 'packets': [...]}}
    """
    # 1. Identify all unique methods for columns
    all_methods = set()
    for data in sa_dict.values():
        all_methods.update(data.get('methods', []))
    
    sorted_methods = sorted(list(all_methods))
    
    # 2. Prepare the table headers
    headers = ["DA"] + sorted_methods
    
    # 3. Prepare the rows
    table_data = []
    # Sorting by integer DA key to keep the table organized
    for da in sorted(sa_dict.keys()):
        row = [f"0x{da:02X}"] # Format as hex (e.g., 0x2A)
        info = sa_dict[da]
        
        for method in sorted_methods:
            sas = get_all_sas_for_method(info, method)
            if sas:
                row.append(format_sa_range(sas))
            elif method in info.get('methods', []):
                row.append("X")
            else:
                row.append("")
        
        table_data.append(row)
    
    # 4. Generate the markdown string
    return tabulate(table_data, headers=headers, tablefmt="github")

In [ ]:
load_contrib('automotive.j1939')
from scapy.contrib.cansocket import CANSocket
from scapy.contrib.automotive.j1939 import (
    J1939Socket,
    j1939_scan,
    j1939_scan_passive,
    j1939_scan_dm,
    DmScanResult
)

If you want to see more details of what's going on during the scans, uncomment the below

In [ ]:
import logging
from scapy.contrib.automotive.j1939 import log_j1939

# Set the level to DEBUG
#log_j1939.setLevel(logging.DEBUG)

Let's define the python-can INTERFACE, CHANNEL, and BITRATE we will be using

In [ ]:
INTERFACE=DEFAULT_INTERFACE
CHANNEL=DEFAULT_CHANNEL
BITRATE=500000

Now we will scan for the current CA (Controller Applications) passively; i.e. by just listening for traffic and seeing what is received.

In [ ]:
with CANSocket(interface=INTERFACE, channel=CHANNEL, bitrate=BITRATE) as csock:
    found = j1939_scan_passive(csock)

for sa in found:
    print("0x{:02X} discovered via passive method (listening for traffic)".format(sa))

print(f"found {len(found)}")

We'll save those CAs we found as `found_passive`

In [ ]:
found_passive = found

## Active Scans

Active scanning involves sending requests on the bus and waiting for responses. We will break this down into several standard J1939 techniques.

### 1. Address Claim Scan

This method broadcasts a Request for Address Claimed (PGN 59904 targeting PGN 60928). According to the J1939 standard, all Controller Applications (CAs) currently on the bus must respond with their own Address Claimed message. This is the most standard way to find active CAs.

In [ ]:
with CANSocket(interface=INTERFACE, channel=CHANNEL, bitrate=BITRATE) as csock:
    results_addr_claim = j1939_scan(csock, force=True, methods=['addr_claim'], bitrate=BITRATE, scan_range=range(1, 254), src_addrs=[0xf1])

for da, info in results_addr_claim.items():
    print("DA=0x{:02X} discovered via addr_claim method ({})".format(
          da, get_method_summary(info, 'addr_claim')))

found_addr_claim = list(results_addr_claim.keys())

### 2. ECU Identification Scan

This method broadcasts a Request for ECU Identification Information (PGN 59904 targeting PGN 64965). Many ECUs will respond with their identification details, which confirms their presence on the bus.

In [ ]:
with CANSocket(interface=INTERFACE, channel=CHANNEL, bitrate=BITRATE) as csock:
    results_ecu_id = j1939_scan(csock, force=True, methods=['ecu_id'], bitrate=BITRATE, scan_range=range(1, 254), src_addrs=[0xf1])

for da, info in results_ecu_id.items():
    for method in info["methods"]:
        print("DA=0x{:02X} discovered via {} method ({})".format(
              da, method, get_method_summary(info, method)))

found_ecu_id = list(results_ecu_id.keys())

### 3. RTS Probe Scan

This method attempts to initiate a Transport Protocol (TP) session with each address in the scan range by sending a "Request to Send" (RTS) frame. If an ECU exists at the target address, it is expected to respond with either a "Clear to Send" (CTS) or a "Connection Abort", confirming its existence.

WARNING: this method is very slow

In [ ]:
with CANSocket(interface=INTERFACE, channel=CHANNEL, bitrate=BITRATE) as csock:
    results_rts_probe = j1939_scan(csock, force=True, methods=['rts_probe'], bitrate=BITRATE, scan_range=range(1, 254), src_addrs=[0xf1])

for da, info in results_rts_probe.items():
    for method in info["methods"]:
        print("DA=0x{:02X} discovered via {} method ({})".format(
              da, method, get_method_summary(info, method)))

found_rts_probe = list(results_rts_probe.keys())

### 4. Unicast Request Scan

Similar to the broadcast Address Claim scan, this method sends a *unicast* Request for Address Claimed to every specific address in the scan range.

WARNING: this method is pretty slow too

In [ ]:
with CANSocket(interface=INTERFACE, channel=CHANNEL, bitrate=BITRATE) as csock:
    results_unicast = j1939_scan(csock, force=True, methods=['unicast'], bitrate=BITRATE, scan_range=range(1, 254), src_addrs=[0xf1])

for da, info in results_unicast.items():
    for method in info["methods"]:
        print("DA=0x{:02X} discovered via {} method ({})".format(
              da, method, get_method_summary(info, method)))

found_unicast = list(results_unicast.keys())

## Discovery Summary

Here is a summary of all J1939 CAs discovered so far through passive and active techniques.

In [ ]:
# Merge all results into a single summary
results_summary = merge_scan_results(
    results_addr_claim, 
    results_ecu_id, 
    results_rts_probe, 
    results_unicast,
    passive_results=found_passive)
print(generate_markdown_table(results_summary))

Let's save all the Controller Application (CA) Destination Addresses (DA) we have seen so far as `candidate_das` we can investigate further below

In [ ]:
candidate_das=list(results_summary.keys())

## Identify UDS ECUs

This method attempts to discover UDS servers on PGN 0xDA by sending a *unicast* "Tester Present" request to each specific address in the scan range.

This will test both 'Physical' direct addressing and 'Functional' broadcast requests for UDS. 

In [ ]:
with CANSocket(interface=INTERFACE, channel=CHANNEL, bitrate=BITRATE) as csock:
    found_uds_physical = j1939_scan(csock, force=True, methods=['uds'], bitrate=BITRATE, scan_range=candidate_das, src_addrs=range(0xf1, 0xfb))

for da, info in found_uds_physical.items():
    for method in info["methods"]:
        print("DA=0x{:02X} discovered via {} method ({})".format(
              da, method, get_method_summary(info, method)))

found_uds = found_uds_physical.keys()

## Identify XCP ECUs

Let's use an 'XCP' scanning technique. Similar to the UDS method, this assumes there is an XCP server on PGN PropA 0xEF and attempts to connect. If a positive response is received, we know the CA supports XCP.

Note: XCP will be on PGN PropA (0xEF) but may be locked to respond only to specific source addresses. Below we try a single guess: `src_addrs=[0xac]`; you may need to expand to other guesses such as `src_addrs=[0xf1, 0xf2, 0xf3]`. To maximize your chances of finding XCP you need to expand the `src_addrs=` to the entire possible set of addresses: `range(245)`.

In [ ]:
with CANSocket(interface=INTERFACE, channel=CHANNEL, bitrate=BITRATE) as csock:
    found = j1939_scan(csock, force=True, methods=['xcp'], bitrate=BITRATE, scan_range=candidate_das, src_addrs=[0xac], diag_pgn=0xEF)

for da, info in found.items():
    for method in info["methods" ]:
        print("DA=0x{:02X} discovered via {} method on PropA 0xEF ({})".format(
              da, method, get_method_summary(info, method)))


Save these CAs as `found_xcp`

In [ ]:
found_xcp=found.keys()

## Identify KWP2000 ECUs (PGN 0xEF)

Some ECUs might expose KWP2000 also. 0xDA is for UDS according to J1939, so other diagnostics will be on Proprietary A (PGN PropA 0xEF).

Just as in the case of XCP: the KWP server may be locked to specific source addresses. Below we try a single guess: `src_addrs=[0xac]`; you may need to expand to other guesses such as `src_addrs=[0xf1, 0xf2, 0xf3]`.  To maximize your changes of finding the KWP server you need to expand the `src_addrs=` to the entire possible set of addresses: `range(245)`.

In [ ]:
with CANSocket(interface=INTERFACE, channel=CHANNEL, bitrate=BITRATE) as csock:
    found = j1939_scan(csock, force=True, methods=['uds'], bitrate=BITRATE, scan_range=candidate_das, src_addrs=[0xac], diag_pgn=0xEF)

for da, info in found.items():
    for method in info["methods"]:
        print("DA=0x{:02X} discovered via {} method on PropA 0xEF ({})".format(
              da, method, get_method_summary(info, method)))

found_kwp = found.keys()

## DM Scanning Each

Another J1939-specific thing is DMs. We can test each of those identified CA addresses for the DMs that they support

In [ ]:
def print_dm_matrix(scan_data):
    """
    Prints a wide table where each row is a Destination Address and each column is a DM.
    """
    if not scan_data:
        print("No scan data provided.")
        return

    # 1. Identify all unique DM names present across all DAs and sort them naturally
    all_dm_names = set()
    for dms in scan_data.values():
        all_dm_names.update(dms.keys())
    
    # Natural sort: DM1, DM2 ... DM10 instead of DM1, DM10, DM2
    sorted_columns = sorted(list(all_dm_names), 
                            key=lambda x: int(x.replace('DM', '')) if x.startswith('DM') else 0)

    # 2. Define column width based on the longest error message or DM name
    col_width = 10 
    da_width = 6

    # 3. Print Header
    header = f"{'DA':<{da_width}} | " + " | ".join(f"{dm:<{col_width}}" for dm in sorted_columns)
    print(header)
    print("-" * len(header))

    # 4. Print Rows
    for da in sorted(scan_data.keys()):
        row_cells = []
        for dm in sorted_columns:
            result = scan_data[da].get(dm)
            
            if result is None:
                cell_text = "-" # DM not probed for this DA
            elif result.supported:
                cell_text = "✓"
            else:
                # Use error string (Timeout, NACK, etc.)
                cell_text = result.error if result.error else "No"
            
            row_cells.append(f"{cell_text:<{col_width}}")
        
        print(f"{da:<{da_width}} | " + " | ".join(row_cells))

# Example Usage:
# print_dm_matrix(your_scan_results)

def print_dm_matrix_transposed(scan_data):
    """
    Prints a transposed DM support table:
    Rows = DM names (DM1, DM2, etc.)
    Columns = Destination Addresses (19, 20, etc.)
    """
    if not scan_data:
        print("No scan data provided.")
        return

    # 1. Get and sort DAs (Columns)
    sorted_das = sorted(scan_data.keys())
    
    # 2. Get and sort all unique DM names (Rows)
    all_dm_names = set()
    for dms in scan_data.values():
        all_dm_names.update(dms.keys())
    
    sorted_dms = sorted(list(all_dm_names), 
                        key=lambda x: int(x.replace('DM', '')) if x.startswith('DM') else 0)

    # 3. Define widths
    dm_col_width = 6
    da_col_width = 12  # Wide enough for "Timeout" or "NACK"

    # 4. Print Header (DA IDs)
    header = f"{'DM':<{dm_col_width}} | " + " | ".join(f"DA {da:<{da_col_width-3}}" for da in sorted_das)
    print(header)
    print("-" * len(header))

    # 5. Print Rows (One row per DM)
    for dm in sorted_dms:
        row_cells = []
        for da in sorted_das:
            result = scan_data[da].get(dm)
            
            if result is None:
                cell_text = "-"
            elif result.supported:
                cell_text = "✓"
            else:
                # Combine supported/error into the cell as requested
                cell_text = result.error if result.error else "No"
            
            row_cells.append(f"{cell_text:<{da_col_width}}")
        
        print(f"{dm:<{dm_col_width}} | " + " | ".join(row_cells))

# Example usage:
# print_dm_matrix_transposed(your_scan_results)

In [ ]:
target_cas = candidate_das
results = {}
with CANSocket(interface=INTERFACE, channel=CHANNEL, bitrate=BITRATE) as csock:
    for da in target_cas:
        results[da] = j1939_scan_dm(csock, target_da=da, dms=["DM1", "DM14"], bitrate=BITRATE)

print_dm_matrix(results)